In [1]:
#Teste 1 — tool customizada mínima
from langchain_core.tools import tool

@tool
def soma(a: int, b: int) -> int:
    """Soma dois números inteiros."""
    return a + b

def test_simple_tool():
    result = soma.invoke({"a": 2, "b": 3})
    assert result == 5
    print("Tool customizada funcionando:", result)

test_simple_tool()

/home/vicrrs/miniconda3/envs/python/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Tool customizada funcionando: 5


In [2]:
!pip install wikipedia
!pip install langchain-community

In [3]:
# Wikipedia wrapper
from langchain_community.utilities import WikipediaAPIWrapper

def test_wikipedia_wrapper():
    wiki = WikipediaAPIWrapper()
    result = wiki.run("LangChain")
    assert isinstance(result, str)
    assert len(result) > 0
    print("WikipediaAPIWrapper funcionando")
    print(result[:500])

test_wikipedia_wrapper()

WikipediaAPIWrapper funcionando
Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: AI agent
Summary: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a cl


In [4]:
# tool da Wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

def test_wikipedia_tool():
    api_wrapper = WikipediaAPIWrapper()
    tool = WikipediaQueryRun(api_wrapper=api_wrapper)

    result = tool.invoke("LangChain")
    assert isinstance(result, str)
    assert len(result) > 0
    print("Wikipedia tool funcionando")
    print(result[:500])

test_wikipedia_tool()

Wikipedia tool funcionando
Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.

Page: AI agent
Summary: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a cl


In [5]:
#Testes com ferramenta customizada
# tool personalizada com regra simples
from langchain_core.tools import tool

@tool
def calcular_area_retangulo(base: float, altura: float) -> float:
    """Calcula a área de um retângulo."""
    return base * altura

def test_custom_tool_area():
    result = calcular_area_retangulo.invoke({"base": 5, "altura": 4})
    assert result == 20
    print("Tool customizada de área funcionando:", result)

test_custom_tool_area()

Tool customizada de área funcionando: 20.0


In [6]:
# tool customizada de texto
from langchain_core.tools import tool

@tool
def resumir_titulo(texto: str) -> str:
    """Retorna um resumo curto em formato de título."""
    return texto[:30] + "..." if len(texto) > 30 else texto

def test_custom_tool_text():
    result = resumir_titulo.invoke({"texto": "LangChain é um framework para aplicações com LLMs"})
    assert isinstance(result, str)
    print("Tool customizada de texto funcionando:", result)

test_custom_tool_text()

Tool customizada de texto funcionando: LangChain é um framework para ...


### Testes com ReAct

ReAct é o padrão de agente que mistura:

* raciocínio

* ação

* observação

Na prática: o agente pensa, escolhe uma tool, executa e volta com resposta.

Como você está usando Hugging Face fora do wrapper tradicional do LangChain, o jeito mais seguro aqui é testar primeiro o conceito de tool calling e depois, se quiser, adaptar para um chat model compatível com agent executor.

In [7]:
# simulação de fluxo ReAct
def test_react_concept():
    pergunta = "Quanto é 7 + 8?"
    pensamento = "Preciso usar uma ferramenta de soma."
    acao = soma.invoke({"a": 7, "b": 8})
    observacao = f"Resultado da ferramenta: {acao}"
    resposta_final = f"A resposta é {acao}."

    assert acao == 15
    print("Simulação ReAct funcionando")
    print("Pergunta:", pergunta)
    print("Pensamento:", pensamento)
    print("Observação:", observacao)
    print("Resposta final:", resposta_final)

test_react_concept()

Simulação ReAct funcionando
Pergunta: Quanto é 7 + 8?
Pensamento: Preciso usar uma ferramenta de soma.
Observação: Resultado da ferramenta: 15
Resposta final: A resposta é 15.


### Criação e execução do agente

Aqui temos duas opções:

* teste real com modelo compatível com agent do LangChain

* teste mockado, que valida o fluxo sem depender da API

In [8]:
# agente mockado simples
tools = {
    "soma": soma,
    "area": calcular_area_retangulo,
}

def fake_agent(user_input: str):
    if "soma" in user_input or "+" in user_input:
        return tools["soma"].invoke({"a": 10, "b": 5})
    if "área" in user_input.lower():
        return tools["area"].invoke({"base": 3, "altura": 6})
    return "Não sei qual ferramenta usar."

def test_fake_agent():
    result1 = fake_agent("faça a soma 10 + 5")
    result2 = fake_agent("calcule a área")

    assert result1 == 15
    assert result2 == 18
    print(" Agente mockado funcionando")
    print("Resultado 1:", result1)
    print("Resultado 2:", result2)

test_fake_agent()

 Agente mockado funcionando
Resultado 1: 15
Resultado 2: 18.0


In [10]:
# Teste básico do modelo
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

def test_hf_basic():
    response = client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[
            {"role": "user", "content": "Explique LangChain em uma frase"}
        ],
        max_tokens=100
    )

    print("Modelo HuggingFace funcionando\n")
    print(response.choices[0].message.content)

test_hf_basic()

/home/vicrrs/miniconda3/envs/python/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo HuggingFace funcionando

O LangChain é uma plataforma de desenvolvimento de inteligência Artificial (IA) que permite criar modelos de linguagem e aplicativos de IA de forma modular e escalável, utilizando uma API única para integrar diferentes componentes e habilidades.


### Criar cliente HuggingFace

In [11]:
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

### Função de chat

Criamos um wrapper para o modelo.

In [12]:
def hf_chat(question: str):

    response = client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[
            {"role": "user", "content": question}
        ],
        max_tokens=200,
        temperature=0.7,
    )

    return response.choices[0].message.content

### Teste básico do modelo

In [14]:
def test_hf_model():

    resposta = hf_chat("Explique LangChain em uma frase")

    print("HuggingFace funcionando\n")
    print(resposta)

test_hf_model()

HuggingFace funcionando

LangChain é uma plataforma de desenvolvimento de inteligência artificial que permite criar modelos de linguagem capazes de aprender e se adaptar a novas tarefas e rotinas, utilizando técnicas de LLM (Large Language Model) e Chain-of-Thought.


### Teste de raciocínio

In [15]:
def test_reasoning():

    pergunta = """
Resolva passo a passo:

12 × 13
"""

    resposta = hf_chat(pergunta)

    print("✅ Teste de raciocínio\n")
    print(resposta)

test_reasoning()

✅ Teste de raciocínio

Vamos resolver a multiplicação de forma passo a passo:

Passo 1: Multiplicar os números 12 e 13.

Para isso, podemos usar a estratégia de multiplicar 12 por 10 e depois adicionar 3 vezes o resultado da multiplicação de 12 por 1.

12 x 10 = 120

Passo 2: Adicionar 3 vezes o resultado da multiplicação de 12 por 1.

12 x 1 = 12

3 vezes 12 = 36

120 + 36 = 156

Passo 3: Resumir o resultado.

12 x 13 = 156


### Teste factual

In [16]:
def test_factual():

    pergunta = "Quem criou o LangChain?"

    resposta = hf_chat(pergunta)

    print("✅ Teste factual\n")
    print(resposta)

test_factual()

✅ Teste factual

Não encontrei informações precisas sobre a criação do LangChain. O LangChain é uma biblioteca de código aberto em Python para construção de aplicativos de chatbot e Gerenciamento de Conhecimento.


### Testes com Tavily

Tavily é um motor de busca para LLMs.

Ele permite que o agente pesquise na internet.

In [17]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.tools.tavily_search import TavilySearchResults

def test_tavily():

    tool = TavilySearchResults(max_results=3)

    result = tool.invoke({"query": "O que é LangChain?"})

    print("Tavily funcionando\n")
    print(result)

test_tavily()

/tmp/ipykernel_41075/4269442108.py:8: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=3)


Tavily funcionando

[{'title': 'O que é o LangChain?', 'url': 'https://aws.amazon.com/pt/what-is/langchain/', 'content': '## O que é o LangChain?\n\nO LangChain é uma estrutura de código aberto para criar aplicações baseadas em grandes modelos de linguagem (LLMs). Os LLMs são grandes modelos de aprendizado profundo pré-treinados em grandes quantidades de dados que podem gerar respostas às consultas do usuário, por exemplo, responder perguntas ou criar imagens a partir de prompts baseados em texto. O LangChain fornece ferramentas e abstrações para melhorar a personalização, a precisão e a relevância das informações que os modelos geram. Por exemplo, os desenvolvedores podem usar componentes do LangChain para criar novas correntes de prompts ou personalizar modelos existentes. O LangChain também inclui componentes que permitem que os LLMs acessem novos conjuntos de dados sem retreinamento. [...] ### Suporte ao desenvolvedor\n\nO LangChain fornece aos desenvolvedores de IA ferramentas par

In [18]:
# Usando Tavily com o modelo

def tavily_agent(question):

    search = TavilySearchResults(max_results=3)

    results = search.invoke({"query": question})

    prompt = f"""
Pergunta: {question}

Resultados da busca:
{results}

Explique a resposta com base nesses resultados.
"""

    return hf_chat(prompt)

In [19]:
# Teste Tavily + LLM

def test_tavily_agent():

    resposta = tavily_agent("Quem criou o LangChain?")

    print("Tavily + HuggingFace\n")
    print(resposta)

test_tavily_agent()

Tavily + HuggingFace

Com base nos resultados da busca, posso responder à sua pergunta sobre quem criou o LangChain.

De acordo com as informações fornecidas pelos resultados da busca, o LangChain foi criado em outubro de 2022 como um projeto de código aberto por Harrison Chase, enquanto trabalhava em uma startup de aprendizado de máquina chamada Robust Intelligence.

Além disso, a Wikipedia menciona que Harrison Chase lançou o LangChain como um projeto de código aberto em outubro de 2022 e que, em abril de 2023, o LangChain havia incorporado e a nova startup havia arrecadado mais de $20 milhões em financiamento de um investidor de risco chamado Sequoia Capital, em uma valoração de pelo menos $200 milhões.

Portanto, podemos concluir que Harrison Chase é o criador do LangChain.
